# Overview

🎶 Welcome to the Missing Fundamental Puzzle! 

Imagine someone playing a beautiful C4 on a violin…
But then—we delete the actual pitch. Poof! 🪄 Gone! 

Yet somehow, your brain still hears C4.   

So here’s the big question:
Can machines hear a musical note that isn’t there? 

In this challenge, every audio clip is a pitch illusion—the true note is missing in action, and only its spectral echoes remain. Despite this, the human brain can still recover the correct pitch by analyzing the spacing and pattern of the remaining harmonics. 

🎯 Your task: Build a model that classifies the original musical note (e.g., C4, G#5) from the given audio—mimicking the brain’s ability to reconstruct pitch from harmonic structure. 

Think you can teach a machine to listen like a human?
Then dive in—and hear the unheard! 🎼🎶🎵

# Description
This is a single-label multiclass audio classification task. 

**Input:** 

Each sample consists of a mono audio waveform in 16-bit PCM WAV format, sampled at 22050 Hz. The duration of each file is variable but typically between 1 and 4 seconds. The audio has been preprocessed and some of the frequencies have been removed from the signal. 

**Target:**

For each audio file, the target is a discrete musical note label representing the original pitch. The label set comprises 61 distinct classes, corresponding to all MIDI notes that appear in the data. 

**Objective:**

Given the modified audio, predict the true underlying note that was originally played. The model must infer pitch from the spectral and temporal structure of the remaining harmonics, without access to the fundamental frequency or other removed harmonics. 

Submissions are evaluated using **standard classification accuracy**:

$$
\text{Accuracy} = \frac{\text{Number of correct predictions}}{\text{Total number of examples}}
$$

🎯 **The higher the score, the better your recommendations!**

## Submission File
For each audio path in the test set, you must predict a encrypted notes pitch label. The file should contain a header and have the following format:

    Path, Pitch_ID
    path/samle1.wav, 0
    path/samle2.wav,1
    path/samle3.wav,2
    etc.

# Dataset description

🎵 Dataset: TinySOL-MF — Missing Fundamental Edition 

This dataset is a specially curated and preprocessed variant of the [TinySOL](https://www.kaggle.com/datasets/thedevastator/tinysol-isolated-musical-notes-from-14-musical-i) dataset, designed to challenge machine learning models with a core phenomenon in auditory perception: pitch perception in the absence of the fundamental frequency. 

🔍 What is the "Missing Fundamental"? 

When a musical note is played, its sound contains a fundamental frequency (f₀) and a series of harmonics (integer multiples of f₀). Remarkably, even if the fundamental frequency is removed, the human brain can still perceive the correct pitch by analyzing the spacing of the remaining harmonics. This psychoacoustic effect is known as the missing fundamental. 

The goal of this competition is to build models that mimic human auditory intelligence—classifying musical notes without relying on the fundamental frequency. 

📁 Dataset Structure 

- Total files: 2,913 isolated musical notes  
- Instruments: 14 orchestral instruments (strings, woodwinds, brass, and keyboard)  
- Split:  
    - Train: 2,330 files (80%)  
    - Test: 583 files (20%)
- Format: 16-bit PCM WAV, mono, 22.05 kHz sampling rate 
- Duration: ~2–4 seconds per note (sustained tones with natural decay)

Every audio file in this dataset has been algorithmically modified.

## Files

*   **train.csv** - the training set
*   **test.csv** - the test set
*   **sample_submission.csv** - a sample submission file in the correct format


## Columns

*   `Path` - path to the audio file
*   `Pitch_ID` - id of a corresponding pitch


```bash
kaggle competitions download -c missing-fundamental-puzzle


In [13]:
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import warnings

DEVICE = 'cuda'

In [22]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

display(train, test)

,Pitch_ID,Path
0,52,train/sample_0.wav
1,64,train/sample_1.wav
2,6,train/sample_2.wav
3,22,train/sample_3.wav
4,52,train/sample_4.wav
...,...,...
2325,50,train/sample_2325.wav
2326,7,train/sample_2326.wav
2327,78,train/sample_2327.wav
2328,48,train/sample_2328.wav


,Path
0,test/sample_0.wav
1,test/sample_1.wav
2,test/sample_2.wav
3,test/sample_3.wav
4,test/sample_4.wav
...,...
578,test/sample_578.wav
579,test/sample_579.wav
580,test/sample_580.wav
581,test/sample_581.wav


In [23]:
N_CLASSES = int(train['Pitch_ID'].max()) + 1

N_CLASSES

82

In [15]:
N_FRAMES = 128   

def to_log_mel(path):
    waveform, _ = librosa.load(path)
    mel = librosa.feature.melspectrogram(y=waveform)
    log_mel = librosa.power_to_db(mel, ref=np.max)

    if log_mel.shape[1] >= N_FRAMES:              
        start = (log_mel.shape[1] - N_FRAMES) // 2
        log_mel = log_mel[:, start:start + N_FRAMES]
    else:                                           
        pad_width = N_FRAMES - log_mel.shape[1]
        log_mel = np.pad(log_mel, ((0, 0), (0, pad_width)), constant_values=log_mel.min())

    return log_mel.astype(np.float32)

In [16]:
X_train = np.stack([to_log_mel(path) for path in train['Path']])
y_train = train['Pitch_ID'].values
X_test = np.stack([to_log_mel(path) for path in test['Path']])

mean = X_train.mean()
std = X_train.std()
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print(X_train.shape, X_test.shape)

(2330, 128, 128) (583, 128, 128)


In [17]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )

model = nn.Sequential(
    nn.BatchNorm2d(1),
    conv_block(1, 32),
    conv_block(32, 64),
    conv_block(64, 128),
    conv_block(128, 128),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Dropout(0.2),
    nn.Linear(128, N_CLASSES),
).to(DEVICE)

In [18]:
EPOCHS = 70
BATCH_SIZE = 64

inputs = torch.tensor(X_train).unsqueeze(1)  
labels = torch.tensor(y_train)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    order = torch.randperm(len(inputs))
    for start in range(0, len(inputs), BATCH_SIZE):
        batch = order[start:start + BATCH_SIZE]
        x = inputs[batch].to(DEVICE)
        y = labels[batch].to(DEVICE)

        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
    scheduler.step()

print('final loss:', loss.item())

C:\Users\Antony\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\triton\windows_utils.py:405: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


final loss: 0.0021757460199296474


In [20]:
model.eval()

with torch.no_grad():
    test_inputs = torch.tensor(X_test).unsqueeze(1).to(DEVICE)
    predictions = model(test_inputs).argmax(dim=1).cpu().numpy()

In [21]:
submission = pd.DataFrame({
    'Path': 'kaggle_dataset/' + test['Path'],
    'Pitch_ID': predictions,
})

submission.to_csv('submission.csv', index=False)
submission.head()

,Path,Pitch_ID
0,kaggle_dataset/test/sample_0.wav,77
1,kaggle_dataset/test/sample_1.wav,69
2,kaggle_dataset/test/sample_2.wav,16
3,kaggle_dataset/test/sample_3.wav,43
4,kaggle_dataset/test/sample_4.wav,38


0.98777 on the leaderboard